In [ ]:
try:
    from openmdao.utils.notebook_utils import notebook_mode  # noqa: F401
except ImportError:
    !python -m pip install openmdao[notebooks]

# BrentSolver

BrentSolver is an implementation of Scipy's Brentq, which is based on the Wijngaarden-Dekker-Brent method. It can solve for a single state bracketed between a lower and upper bound using the bisection method, where the value at the two bounding points must have opposite signs. Convergence is guaranteed for generally well-behaved problems on the interval. Derivatives are not required for this solver, though it is limited to solving a single state. Brent can be nested with other solvers, including other Brent solvers.


## BrentSolver Options

In [ ]:
import openmdao.api as om
om.show_options_table("openmdao.solvers.nonlinear.brent.BrentSolver")


## BrentSolver Constructor

The call signature for the `BrentSolver` constructor is:

```{eval-rst}
    .. automethod:: openmdao.solvers.nonlinear.brent.BrentSolver.__init__
        :noindex:
```

## BrentSolver Example

The following simple example shows the use of the Brent solver to find the output of an `ImplicitComponent` that models the equation `x = a*x**n + b*x - c`. 

In [ ]:
import openmdao.api as om

class CompTest(om.ImplicitComponent):

    def setup(self):
        self.add_input('a', val=1.)
        self.add_input('b', val=1.)
        self.add_input('c', val=10.)
        self.add_input('n', val=77.0/27.0)

        self.add_output('x', val=2., lower=0, upper=100)

    def apply_nonlinear(self, inputs, outputs, residuals, discrete_inputs=None, discrete_outputs=None):
        a = inputs['a']
        b = inputs['b']
        c = inputs['c']
        n = inputs['n']
        x = outputs['x']

        # Can't take fractional power of negative number
        if x >= 0.0:
            fact = x ** n
        else:
            fact = - (-x) ** n

        residuals['x'] = a * fact + b * x - c


prob = om.Problem()
model = prob.model
model.add_subsystem('comp', CompTest(), promotes=['*'])
model.nonlinear_solver = om.BrentSolver(
    state_target='x',
    lower_bound=0.0,
    upper_bound=80.0,
    maxiter=100,
    atol=1e-8,
    rtol=1e-8,
)

prob.setup()
prob.set_solver_print(2)

prob.run_model()

In [ ]:
print(prob.get_val('x'))

In [ ]:
from openmdao.utils.assert_utils import assert_near_equal

assert_near_equal(prob.get_val('x')[0], 2.06720359226, 1e-6)

The convergence history clearly shows the bisection and bracketing process, where the model is evaluated at closer points to the eventual solution.

## BrentSolver Option Example: Upper and Lower Bounds from the Model

The BrentSolver also allows the lower and upper bounds to be sourced from the model output, which is useful in cases where the interval of interest can be determined ahead of time by an upstream calculation. Note that the bounds are only queried at the start of the Brent iteration.

In [ ]:
import openmdao.api as om

prob = om.Problem()
model = prob.model

model.add_subsystem('lower', om.ExecComp('low = 2*a'), promotes=['*'])
model.add_subsystem('upper', om.ExecComp('high = 2*b'), promotes=['*'])

model.add_subsystem('comp', CompTest(), promotes=['*'])
model.nonlinear_solver = om.BrentSolver(
    state_target='x',
    maxiter=100,
    atol=1e-8,
    rtol=1e-8,
    lower_bound_target='low',
    upper_bound_target='high',
)

prob.setup()
prob.set_solver_print(2)

prob.setup()

prob.set_val('a', -5.0)
prob.set_val('b', 55.0)

prob.run_model()

In [ ]:
print(prob.get_val('x'))

In [ ]:
from openmdao.utils.assert_utils import assert_near_equal
assert_near_equal(prob.get_val('x')[0], -3.7451537261581453, 1e-6)